In [4]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from scipy.spatial.distance import pdist

In [5]:
training_configs = np.load("LJ13_configs.npy")
testing_configs = np.load("LJ13_configs_2.npy")

In [7]:
com = training_configs.mean(axis=1, keepdims=True)
com_2 = testing_configs.mean(axis=1, keepdims=True)
training_configs_centered = training_configs - com
testing_configs_centered = testing_configs - com_2

In [8]:
training_configs_centered_tensor = torch.tensor(training_configs_centered, dtype=torch.float32)
testing_configs_centered_tensor = torch.tensor(testing_configs_centered, dtype=torch.float32)
print(training_configs_centered_tensor.shape)
print(testing_configs_centered_tensor.shape)

torch.Size([24994, 13, 3])
torch.Size([4994, 13, 3])


In [9]:
class MessageMLP(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2*hidden_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU()
        )

    def forward(self, x):
        return self.net(x)

In [10]:
class FeatureMLP(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2*hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, x):
        return self.net(x)

In [11]:
class CoordMLP(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2*hidden_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

        torch.nn.init.xavier_uniform_(
            self.net[-1].weight,
            gain=0.001
        )

        torch.nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(self, x):
        return self.net(x)

In [12]:
class EGNNLayer(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.message_mlp = MessageMLP(hidden_dim)

        self.feature_mlp = FeatureMLP(hidden_dim)

        self.coord_mlp = CoordMLP(hidden_dim)

    def forward(self, x, h, log_sigma):

        N = x.shape[1]

        diff = x[:,:,None,:] - x[:,None,:,:]
        dist2 = (diff**2).sum(dim=-1, keepdim=True)

        hi = h[:,:,None,:]
        hi = hi.expand(-1,-1,N,-1)

        hj = h[:,None,:,:]
        hj = hj.expand(-1,N,-1,-1)

        log_sigma_pair = log_sigma[:,None,None,:]
        log_sigma_pair = log_sigma_pair.expand(-1,N,N,-1)

        message_input = torch.cat([hi, hj, dist2, log_sigma_pair], dim=-1)

        messages = self.message_mlp(message_input)

        agg_messages = messages.sum(dim=2)

        feature_input = torch.cat([h, agg_messages], dim=-1)

        feature_update = self.feature_mlp(feature_input)

        h_new = h + feature_update

        coord_weights = self.coord_mlp(message_input)

        coord_update = diff * coord_weights

        delta_x = coord_update.sum(dim=2)

        delta_x = delta_x / x.shape[1]

        x_new = x + delta_x

        return x_new, h_new, delta_x

    

In [13]:
class ScoreNetwork(nn.Module):

    def __init__(
        self,
        hidden_dim=64,
        num_layers=3
    ):
        super().__init__()

        self.embedding = nn.Linear(1, hidden_dim)

        self.layers = nn.ModuleList(
            [
                EGNNLayer(hidden_dim)
                for _ in range(num_layers)
            ]
        )


    def forward(self, x, log_sigma):

        atom_features = torch.ones(x.shape[0], x.shape[1], 1)

        h = self.embedding(atom_features)

        f_pred = torch.zeros_like(x)

        for layer in self.layers:

            x, h, delta_x = layer(x, h, log_sigma)

            f_pred = f_pred + delta_x

        f_pred = f_pred - f_pred.mean(dim=1, keepdim=True)


        return f_pred


In [14]:
X = training_configs_centered.reshape(training_configs_centered.shape[0],39)
X = torch.tensor(X, dtype=torch.float32)

X_np = X.cpu().numpy()

sigma_max = pdist(X_np).max()
sigma_min = 0.01

print("sigma_max:", sigma_max)

sigma_max: 6.440439591617135


In [15]:
def get_sigma(t):

    return sigma_min * (sigma_max/sigma_min)**t

In [16]:
x_train = training_configs_centered_tensor[:20000]
x_test = testing_configs_centered_tensor

In [167]:
def train_model(
  optimizer_name="Adam",
  lr = 1e-4,
  batch_size = 100,
  width = 64,
  depth = 3,
  num_epochs = 600   
):
    dataset = TensorDataset(x_train)

    loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)

    model = ScoreNetwork(hidden_dim=width,num_layers=depth)

    if optimizer_name == "Adam":
        
        optimizer=optim.Adam(
            model.parameters(),
            lr=lr
        )
    else:

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr
        )
    

    loss_history = []

    for epoch in range(num_epochs):
        
        epoch_loss = 0.0

        for (x_batch,) in loader:
            
            batch_size_current = x_batch.shape[0]
            
            t = torch.rand(batch_size_current,1)

            sigma = get_sigma(t)

            sigma_expanded = sigma.view(-1,1,1)

            z = torch.randn_like(x_batch)

            z = z - z.mean(dim=1, keepdim=True)
            
            x_noisy = x_batch + sigma_expanded * z

            log_sigma = torch.log(sigma)
            
            target = -z  

            f_pred = model(x_noisy, log_sigma)

            loss_per_sample = ((f_pred - target) ** 2).sum(dim=(1,2))

            #weights = sigma.squeeze() ** 2

            loss = (loss_per_sample).mean()

            optimizer.zero_grad()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        loss_history.append(avg_loss)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:4d} | Loss = {avg_loss:.6f}")

    return model, loss_history

In [168]:
model, loss_history = train_model()
torch.save(model.state_dict(), "lj13_f_theta_memorization_test.pth")

Epoch    0 | Loss = 25.494727
Epoch   10 | Loss = 15.025975
Epoch   20 | Loss = 14.122591
Epoch   30 | Loss = 14.046001
Epoch   40 | Loss = 13.606289
Epoch   50 | Loss = 13.752855
Epoch   60 | Loss = 13.387819
Epoch   70 | Loss = 13.221796
Epoch   80 | Loss = 12.983873
Epoch   90 | Loss = 12.928544
Epoch  100 | Loss = 12.765292
Epoch  110 | Loss = 12.755104
Epoch  120 | Loss = 12.588436
Epoch  130 | Loss = 12.522696
Epoch  140 | Loss = 12.571121
Epoch  150 | Loss = 12.555610
Epoch  160 | Loss = 12.530142
Epoch  170 | Loss = 12.517880
Epoch  180 | Loss = 12.594250
Epoch  190 | Loss = 12.317362
Epoch  200 | Loss = 12.584187
Epoch  210 | Loss = 12.252126
Epoch  220 | Loss = 12.362227
Epoch  230 | Loss = 12.312278
Epoch  240 | Loss = 12.489959
Epoch  250 | Loss = 12.312463
Epoch  260 | Loss = 12.432785
Epoch  270 | Loss = 12.456615
Epoch  280 | Loss = 12.298976
Epoch  290 | Loss = 12.405964
Epoch  300 | Loss = 12.363281
Epoch  310 | Loss = 12.277277
Epoch  320 | Loss = 12.380791
Epoch  330

In [107]:
torch.save(model.state_dict(), "lj13_f_theta_memorization_test.pth")

In [17]:
model = ScoreNetwork(hidden_dim=64, num_layers=3)

model.load_state_dict(torch.load("lj13_f_theta_memorization_test.pth"))

model.eval()

ScoreNetwork(
  (embedding): Linear(in_features=1, out_features=64, bias=True)
  (layers): ModuleList(
    (0-2): 3 x EGNNLayer(
      (message_mlp): MessageMLP(
        (net): Sequential(
          (0): Linear(in_features=130, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
          (3): SiLU()
        )
      )
      (feature_mlp): FeatureMLP(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
        )
      )
      (coord_mlp): CoordMLP(
        (net): Sequential(
          (0): Linear(in_features=130, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
          (3): SiLU()
          (4): Linear(in_features=64, out_features=1, bias=True)
        )
      )
    )
  )
)

In [18]:
def lj_energy(coords):

    N = coords.shape[0]

    energy = 0.0

    for i in range(N):
        for j in range(i + 1, N):

            r = torch.norm(coords[i] - coords[j])

            inv_r6 = (1.0 / r) ** 6
            inv_r12 = inv_r6 ** 2

            energy += 4 * (inv_r12 - inv_r6)

    return energy

In [19]:
def lj_energy_batch(coords):
    # coords shape: [batch, N, 3]

    diff = coords[:, :, None, :] - coords[:, None, :, :]

    # Pairwise distances
    r = torch.norm(diff, dim=-1)

    # Only take i < j
    i, j = torch.triu_indices(
        coords.shape[1],
        coords.shape[1],
        offset=1,
        device=coords.device
    )

    r_pairs = r[:, i, j]

    inv_r6 = (1.0 / r_pairs) ** 6
    inv_r12 = inv_r6 ** 2

    energy = 4 * (inv_r12 - inv_r6)

    # Sum over all 78 pairs
    return energy.sum(dim=1)

In [20]:
import torch.nn.functional as F

In [21]:
n_train_configs = 500
n_test_configs = 500
n_noise = 100
Temp = 0.1
beta = 1.0 / Temp
x0_train = x_train[:n_train_configs]
x0_test = x_test[:n_test_configs]

sigma_values = [0.1, 0.05, 0.02, 0.01]

In [22]:
results_train = []

for sigma_value in sigma_values:

    sigma = torch.tensor(sigma_value)

    cos_noise_all = []
    cos_physics_all = []
    norm_ratio_noise_all = []
    norm_ratio_physics_all = []

    for repeat in range(n_noise):


        z = torch.randn_like(x0_train)
        z = z - z.mean(dim=1, keepdim=True)


        x_tilde = x0_train + sigma * z


        log_sigma = torch.full((n_train_configs, 1), np.log(sigma_value))

        with torch.no_grad():
            f_pred = model(x_tilde, log_sigma)

        score_model = f_pred / sigma
        score_noise = -z / sigma


        x_phys = (x_tilde.detach().clone().requires_grad_(True))

        energies = lj_energy_batch(x_phys)
        total_energy = energies.sum()

        grad_U = torch.autograd.grad(total_energy, x_phys)[0]

        score_physics = -beta * grad_U


        model_flat = score_model.reshape(n_train_configs, -1)
        noise_flat = score_noise.reshape(n_train_configs, -1)
        physics_flat = score_physics.reshape(n_train_configs, -1)

        cos_noise = F.cosine_similarity(model_flat, noise_flat, dim=1)

        cos_physics = F.cosine_similarity(model_flat, physics_flat, dim=1)

        model_norm = torch.norm(model_flat, dim=1)
        noise_norm = torch.norm(noise_flat, dim=1)
        physics_norm = torch.norm(physics_flat, dim=1)

        norm_ratio_noise = model_norm / noise_norm
        norm_ratio_physics = model_norm / physics_norm

        cos_noise_all.extend(cos_noise.detach().cpu().numpy())

        cos_physics_all.extend(cos_physics.detach().cpu().numpy())

        norm_ratio_noise_all.extend(norm_ratio_noise.detach().cpu().numpy())
        norm_ratio_physics_all.extend(norm_ratio_physics.detach().cpu().numpy())


    results_train.append({
        "sigma": sigma_value,
        "cos_noise": np.mean(cos_noise_all),
        "cos_physics": np.mean(cos_physics_all),
        "norm_ratio_noise": np.mean(norm_ratio_noise_all),
        "norm_ratio_physics": np.mean(norm_ratio_physics_all)
    })

In [23]:
for result in results_train:
    print(
        f"sigma = {result['sigma']:.4f}, "
        f"cos_noise = {result['cos_noise']:.4f}, "
        f"cos_physics = {result['cos_physics']:.4f}, "
        f"norm_noise = {result['norm_ratio_noise']:.4f}, "
        f"norm_physics = {result['norm_ratio_physics']:.4f}"
    )

sigma = 0.1000, cos_noise = 0.8986, cos_physics = 0.5035, norm_noise = 0.8820, norm_physics = 0.0112
sigma = 0.0500, cos_noise = 0.7885, cos_physics = 0.6905, norm_noise = 0.7826, norm_physics = 0.0974
sigma = 0.0200, cos_noise = 0.5276, cos_physics = 0.9238, norm_noise = 0.5241, norm_physics = 0.4489
sigma = 0.0100, cos_noise = 0.3201, cos_physics = 0.9812, norm_noise = 0.3315, norm_physics = 0.7637


In [24]:
results_test = []

for sigma_value in sigma_values:

    sigma = torch.tensor(sigma_value)

    cos_noise_all = []
    cos_physics_all = []
    norm_ratio_noise_all = []
    norm_ratio_physics_all = []

    for repeat in range(n_noise):


        z = torch.randn_like(x0_test)
        z = z - z.mean(dim=1, keepdim=True)


        x_tilde = x0_test + sigma * z


        log_sigma = torch.full((n_test_configs, 1), np.log(sigma_value))

        with torch.no_grad():
            f_pred = model(x_tilde, log_sigma)

        score_model = f_pred / sigma
        score_noise = -z / sigma


        x_phys = (x_tilde.detach().clone().requires_grad_(True))

        energies = lj_energy_batch(x_phys)
        total_energy = energies.sum()

        grad_U = torch.autograd.grad(total_energy, x_phys)[0]

        score_physics = -beta * grad_U


        model_flat = score_model.reshape(n_test_configs, -1)
        noise_flat = score_noise.reshape(n_test_configs, -1)
        physics_flat = score_physics.reshape(n_test_configs, -1)

        cos_noise = F.cosine_similarity(model_flat, noise_flat, dim=1)

        cos_physics = F.cosine_similarity(model_flat, physics_flat, dim=1)

        model_norm = torch.norm(model_flat, dim=1)
        noise_norm = torch.norm(noise_flat, dim=1)
        physics_norm = torch.norm(physics_flat, dim=1)

        norm_ratio_noise = model_norm / noise_norm
        norm_ratio_physics = model_norm / physics_norm

        cos_noise_all.extend(cos_noise.detach().cpu().numpy())

        cos_physics_all.extend(cos_physics.detach().cpu().numpy())

        norm_ratio_noise_all.extend(norm_ratio_noise.detach().cpu().numpy())
        norm_ratio_physics_all.extend(norm_ratio_physics.detach().cpu().numpy())


    results_test.append({
        "sigma": sigma_value,
        "cos_noise": np.mean(cos_noise_all),
        "cos_physics": np.mean(cos_physics_all),
        "norm_ratio_noise": np.mean(norm_ratio_noise_all),
        "norm_ratio_physics": np.mean(norm_ratio_physics_all)
    })

In [25]:
for result in results_test:
    print(
        f"sigma = {result['sigma']:.4f}, "
        f"cos_noise = {result['cos_noise']:.4f}, "
        f"cos_physics = {result['cos_physics']:.4f}, "
        f"norm_noise = {result['norm_ratio_noise']:.4f}, "
        f"norm_physics = {result['norm_ratio_physics']:.4f}"
    )

sigma = 0.1000, cos_noise = 0.8989, cos_physics = 0.5030, norm_noise = 0.8823, norm_physics = 0.0112
sigma = 0.0500, cos_noise = 0.7885, cos_physics = 0.6900, norm_noise = 0.7832, norm_physics = 0.0970
sigma = 0.0200, cos_noise = 0.5281, cos_physics = 0.9239, norm_noise = 0.5242, norm_physics = 0.4494
sigma = 0.0100, cos_noise = 0.3218, cos_physics = 0.9814, norm_noise = 0.3310, norm_physics = 0.7654


The second memorization test also confirms that the network is not memorizing. The training and test results are approximately the same. 